## Setup

In [1]:
from pathlib import Path

# Prefer the new package when available (LangChain deprecation path)
try:
    from langchain_chroma import Chroma
except Exception:
    from langchain_community.vectorstores import Chroma

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.language_models.llms import LLM
from typing import Optional, List, Any

# Define paths (relative to this notebook's folder)
VECTORSTORE_DIR = Path("../data/vectorstore")

print(f"📂 Vector store location: {VECTORSTORE_DIR.resolve()}")
print(f"📊 Vector store exists: {VECTORSTORE_DIR.exists()}")

/Users/adel/Desktop/Rag-project/rag-env/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
/Users/adel/Desktop/Rag-project/rag-env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


📂 Vector store location: /Users/adel/Desktop/Rag-project/data/vectorstore
📊 Vector store exists: True


## RAG Functions

In [2]:
def load_vectorstore(persist_dir, embedding_model, device: str = "mps"):
    """Load existing vector store from disk.

    embedding_model MUST match ingestion; otherwise distances are meaningless.
    """
    embeddings = HuggingFaceEmbeddings(
        model_name=embedding_model,
        model_kwargs={"device": device},
        encode_kwargs={"batch_size": 32},
    )

    vectorstore = Chroma(
        embedding_function=embeddings,
        persist_directory=str(persist_dir),
        collection_name="rag",
    )

    return vectorstore


def format_context(docs):
    """Format retrieved documents as plain text context (strips metadata)"""
    context = ""
    for i, doc in enumerate(docs, 1):
        context += f"\n[Document {i}]\n{doc.page_content}\n"
    return context


class LlamaCppDirectLLM(LLM):
    """Thin LangChain LLM wrapper around a pre-loaded llama_cpp.Llama instance."""
    llm_instance: Any
    max_tokens: int = 512
    temperature: float = 0.2
    stop: List[str] = []

    class Config:
        arbitrary_types_allowed = True

    def _call(self, prompt: str, stop: Optional[List[str]] = None, **kwargs) -> str:
        result = self.llm_instance(
            prompt,
            max_tokens=self.max_tokens,
            temperature=self.temperature,
            repeat_penalty=1.1,
            stop=stop or self.stop
        )
        return result["choices"][0]["text"].strip()

    @property
    def _llm_type(self) -> str:
        return "llama_cpp_direct"


def retrieve_with_scores(vectorstore, query: str, k: int = 5):
    """Retrieve with numeric scores.

    For Chroma in LangChain, the score is typically a *distance* (lower is better).
    """
    return vectorstore.similarity_search_with_score(query, k=k)


def format_context_with_citations(docs_with_scores, max_chars_per_chunk: int = 1200) -> str:
    """Format retrieved docs as context, including source/page and a short excerpt."""
    parts = []
    for i, (doc, score) in enumerate(docs_with_scores, 1):
        source = doc.metadata.get("source", "unknown")
        page = doc.metadata.get("page", "?")
        text = (doc.page_content or "").strip().replace("\u0000", "")
        if len(text) > max_chars_per_chunk:
            text = text[:max_chars_per_chunk].rsplit(" ", 1)[0] + " …"

        parts.append(
            f"[Source {i}] {source} (page {page}) | score={score:.4f}\n"
            f"\"\"\"{text}\"\"\""
        )

    return "\n\n".join(parts)


def create_rag_prompt():
    template = """[INST] You are a careful assistant.

You must follow these rules:
- Use ONLY the provided sources.
- If the sources do not contain the answer, reply exactly: NOT_FOUND
- Do NOT use outside knowledge.
- Do NOT provide external links.
- After your answer, add a section 'Citations' with 1-3 bullet points, each quoting an exact phrase from the sources and referencing [Source N].

Sources:
{context}

Question: {question} [/INST]"""

    return PromptTemplate.from_template(template)


def create_llm():
    from llama_cpp import Llama

    print("⏳ Loading Mistral-7B-Instruct-v0.3 (Q4_K_M)...")
    llm_instance = Llama.from_pretrained(
        repo_id="bartowski/Mistral-7B-Instruct-v0.3-GGUF",
        filename="Mistral-7B-Instruct-v0.3-Q4_K_M.gguf",
        n_ctx=4096,
        n_gpu_layers=-1,
        n_batch=512,
        n_threads=10,
        verbose=False
    )
    print("✅ Model loaded successfully!")

    return LlamaCppDirectLLM(
        llm_instance=llm_instance,
        max_tokens=384,
        temperature=0.0,
        stop=["</s>"]
    )

/var/folders/87/zmqs14g943d92b6n0nm7_2qw0000gn/T/ipykernel_55546/3340215635.py:25: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  class LlamaCppDirectLLM(LLM):


## Initialize RAG Pipeline

In [3]:
print("🔄 Loading vector store...")
vectorstore = load_vectorstore(
    persist_dir=VECTORSTORE_DIR,
    embedding_model="sentence-transformers/all-MiniLM-L6-v2"
)
print("✅ Vector store loaded!")

print("\n🔄 Creating prompt + LLM...")
prompt = create_rag_prompt()
llm = create_llm()
print("✅ Ready!")

🔄 Loading vector store...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9928.60it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/var/folders/87/zmqs14g943d92b6n0nm7_2qw0000gn/T/ipykernel_55546/3340215635.py:9: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


✅ Vector store loaded!

🔄 Creating prompt + LLM...
⏳ Loading Mistral-7B-Instruct-v0.3 (Q4_K_M)...


/Users/adel/Desktop/Rag-project/rag-env/lib/python3.14/site-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
llama_context: n_ctx_seq (4096) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


✅ Model loaded successfully!
✅ Ready!


## Query & Generate Answers

In [4]:
# Diagnostic: Check vectorstore
print("🔍 Vectorstore Diagnostics")
print(f"   - Collection count: {vectorstore._collection.count()}")
print("   - Embedding model: sentence-transformers/all-MiniLM-L6-v2")

# Try a scored retrieval (lower score = better match)
test_query = "bioinformatics"
test_hits = retrieve_with_scores(vectorstore, test_query, k=5)
print(f"   - Test search for '{test_query}': {len(test_hits)} hits")
if test_hits:
    for i, (doc, score) in enumerate(test_hits[:2], 1):
        print(f"      [{i}] score={score:.4f} | {doc.page_content[:100]}...")

🔍 Vectorstore Diagnostics
   - Collection count: 3262
   - Embedding model: sentence-transformers/all-MiniLM-L6-v2
   - Test search for 'bioinformatics': 5 hits
      [1] score=1.0574 | sequencingtechnologyfor
tumorinput.
--RGPL-tumor
RGPU Specifiesthereadgroup
platformunit.
--RGPU
RGP...
      [2] score=1.0574 | sequencingtechnologyfor
tumorinput.
--RGPL-tumor
RGPU Specifiesthereadgroup
platformunit.
--RGPU
RGP...


In [8]:
# Practical grounding: refuse to answer when retrieval is weak.
# NOTE: For Chroma, scores are typically distances where lower = more similar.
# Distances depend on embedding model + collection contents; tune after you reset/rebuild.
MAX_DISTANCE = 1.4
TOP_K = 5


def ask(question: str) -> str:
    hits = retrieve_with_scores(vectorstore, question, k=TOP_K)

    # Gate: if nothing is close enough, refuse.
    best_score = min((score for _, score in hits), default=float("inf"))
    good_hits = [(doc, score) for (doc, score) in hits if score <= MAX_DISTANCE]

    if not good_hits:
        answer = "NOT_FOUND"
        retrieved_docs = [doc for (doc, _) in hits]
    else:
        # Use only the good hits as sources.
        context = format_context_with_citations(good_hits)
        prompt_text = prompt.format(context=context, question=question)
        try:
            answer = llm.invoke(prompt_text)
        except Exception as e:
            answer = f"Error: {str(e)}"
        retrieved_docs = [doc for (doc, _) in good_hits]

    # Normalize NOT_FOUND to a friendly user-facing answer.
    if isinstance(answer, str) and answer.strip() == "NOT_FOUND":
        answer = (
            "I can’t find that answer in the provided document(s). "
            "Try rephrasing or adding the relevant document to `data/documents/`."
        )

    # Print answer first
    print(f"\nQuestion\n {question}\n")
    print(f"{answer}\n")

    # Then print sources + retrieval confidence
    print(f"Retrieval\n - best_score={best_score:.4f} (lower is better)\n - threshold={MAX_DISTANCE:.2f}\n")

    if retrieved_docs:
        print(f"📚 Sources ({len(retrieved_docs)}):")
        for i, doc in enumerate(retrieved_docs, 1):
            source = doc.metadata.get("source", "unknown")
            page = doc.metadata.get("page", "?")
            print(f"   [{i}] {source} (page {page})")

    return answer


# Example usage
ask("What is the main topic of the document?")
ask("What key information is provided?")
ask("Summarize the content in one paragraph.")


Question
 What is the main topic of the document?

I can’t find that answer in the provided document(s). Try rephrasing or adding the relevant document to `data/documents/`.

Retrieval
 - best_score=1.4987 (lower is better)
 - threshold=1.40

📚 Sources (5):
   [1] ../data/documents/200033181_02_dragen-bio-it-platform-v4-2-doc.pdf (page 32)
   [2] ../data/documents/200033181_02_dragen-bio-it-platform-v4-2-doc.pdf (page 32)
   [3] ../data/documents/200033181_02_dragen-bio-it-platform-v4-2-doc.pdf (page 120)
   [4] ../data/documents/200033181_02_dragen-bio-it-platform-v4-2-doc.pdf (page 120)
   [5] ../data/documents/200033181_02_dragen-bio-it-platform-v4-2-doc.pdf (page 169)

Question
 What key information is provided?

The provided sources contain information about the Illumina DRAGEN Bio-IT Platform v4.2, a tool for analyzing genetic data. Here are some key points:

1. The platform can process input data to generate a reference if one is not already available. This can be done using th

"The provided documents detail the Illumina DRAGEN Bio-IT Platform version 4.2, a tool designed for research purposes and not for diagnostic procedures. One of the main sections discussed is the Mapping/Aligning process, where a total of 816360354 reads were input. Among these reads, there were 15779031 marked as duplicate reads (not removed), representing 1.93% of the total. Additionally, there's a mention of an option called PON, which can be used multiple times or once per file. This option requires a text file that indicates normal count files per line, specifying paths to the list of reference target counts files to be used as a panel of normals (new lines separated)."

## RAG Pipeline Complete
✅ **Full RAG System Active**

Your retrieval-augmented generation pipeline is now fully operational:
- **Document Ingestion**: Completed in `01_ingest.ipynb`
- **Retrieval Setup**: Tested in `02_embed.ipynb`
- **Query Generation**: Now running in `03_rag_pipeline.ipynb`

The system can now answer questions by:
1. Finding relevant documents from your corpus
2. Passing them to an LLM for context
3. Generating accurate answers based on your documents

In [6]:
ask("How do I find the current directory?")  # should refuse unless the docs mention it


Question
 How do I find the current directory?

I can’t find that answer in the provided document(s). Try rephrasing or adding the relevant document to `data/documents/`.

Retrieval
 - best_score=1.4162 (lower is better)
 - threshold=0.45

📚 Sources (5):
   [1] ../data/documents/200033181_02_dragen-bio-it-platform-v4-2-doc.pdf (page 355)
   [2] ../data/documents/200033181_02_dragen-bio-it-platform-v4-2-doc.pdf (page 355)
   [3] ../data/documents/200033181_02_dragen-bio-it-platform-v4-2-doc.pdf (page 34)
   [4] ../data/documents/200033181_02_dragen-bio-it-platform-v4-2-doc.pdf (page 34)
   [5] ../data/documents/200033181_02_dragen-bio-it-platform-v4-2-doc.pdf (page 491)


'I can’t find that answer in the provided document(s). Try rephrasing or adding the relevant document to `data/documents/`.'